In [26]:
from datetime import datetime, timedelta
from binance.um_futures import UMFutures
from database.adatabase import ADatabase
from time import sleep
import pandas as pd
import random
import matplotlib.pyplot as plt
import pytz
kst_timezone = pytz.timezone('Asia/Seoul')

In [27]:
db = ADatabase("sapling")
db.cloud_connect()
bots = db.retrieve("crypto_bots")
keys = db.retrieve("crypto_secrets")
parameter = db.retrieve("crypto_parameter")
db.disconnect()
ticker = parameter["ticker"].item()
band = parameter["band"].item()
stoploss = parameter["stoploss"].item()
profittake = parameter["profittake"].item()
leverage = parameter["leverage"].item()
for bot in bots.iterrows():
    user = bot[1]["username"]
    live = bot[1]["live"]
    if live == True:
        secret = keys[keys["username"]==user]["bsecret"].item()
        key = keys[keys["username"]==user]["bkey"].item()
        umf = UMFutures(key,secret)
        account = umf.account()
        trades = umf.get_account_trades("XRPUSDT")
        trades = pd.DataFrame(trades)
        print(account["totalUnrealizedProfit"])
        positions = pd.DataFrame(account["positions"])
        xrp_positions = positions[positions["symbol"]==ticker]
        orders = pd.DataFrame(umf.get_all_orders("XRPUSDT"))
        new_orders = orders[orders["status"]=="NEW"]

0.00000000


In [28]:
new_orders

,orderId,symbol,status,clientOrderId,price,avgPrice,origQty,executedQty,cumQuote,timeInForce,...,workingType,priceMatch,selfTradePreventionMode,goodTillDate,priceProtect,origType,time,updateTime,activatePrice,priceRate


In [29]:
trades["date"] = [datetime.utcfromtimestamp(float(x) // 1000.0).replace(tzinfo=pytz.utc).astimezone(kst_timezone) for x in trades["time"]]

In [30]:
trades["realizedPnl"] = [float(x) for x in trades["realizedPnl"]]
trades["commission"] = [float(x) for x in trades["commission"]]
trades["price"] = [float(x) for x in trades["price"]]
trades["w/l"] = ["W" if x > 0 else "L" if x < 0 else "N" for x in trades["realizedPnl"]]
trades = trades[trades["date"]>datetime(2024,2,26).replace(tzinfo=pytz.utc).astimezone(kst_timezone)]
trades = trades.groupby(["date","w/l"]).agg({"price":"mean","commission":"sum","realizedPnl":"sum"}).reset_index()
trades["agg_pnl"] = trades["realizedPnl"].cumsum()
trades["agg_commission"] = trades["commission"].cumsum()
trades["pnl"] = trades["agg_pnl"] - trades["agg_commission"]
trades["entry_price"] = trades["price"].shift(1)
trades["price_diff"] = trades["price"].pct_change() * 100
trades = trades[trades["w/l"]!="N"]
trades[["date","price","price_diff","w/l","realizedPnl","agg_pnl","commission","agg_commission","pnl"]]

,date,price,price_diff,w/l,realizedPnl,agg_pnl,commission,agg_commission,pnl
1,2024-02-26 11:58:42+09:00,0.53875,-0.434300,W,0.97545,0.97545,0.105027,0.210541,0.764909
3,2024-02-26 12:16:31+09:00,0.54000,0.092678,W,0.20250,1.17795,0.043740,0.363530,0.814420
5,2024-02-26 12:35:45+09:00,0.54120,0.240785,L,-0.52650,0.65145,0.109593,0.582453,0.068997
7,2024-02-26 12:36:02+09:00,0.54110,0.018484,L,-0.03900,0.61245,0.105514,0.793462,-0.181012
11,2024-02-26 12:40:55+09:00,0.54100,0.036982,L,-0.07500,0.53745,0.101437,1.199175,-0.661725
13,2024-02-26 12:41:06+09:00,0.54120,0.036969,L,-0.07200,0.46545,0.097416,1.393971,-0.928521
15,2024-02-26 12:41:24+09:00,0.54120,0.018481,L,-0.03600,0.42945,0.097416,1.588785,-1.159335
17,2024-02-26 12:41:41+09:00,0.54120,0.018481,L,-0.03600,0.39345,0.097416,1.783599,-1.390149
19,2024-02-26 12:41:58+09:00,0.54120,0.018481,L,-0.03450,0.35895,0.093357,1.970296,-1.611345
21,2024-02-26 12:42:15+09:00,0.54120,0.018481,L,-0.03450,0.32445,0.093357,2.156992,-1.832542
